### Create tools for agentic app to act as a data analyst for the Marketing project data

This project is a **Databricks-native application** and requires a Databricks workspace to run. The code will not execute outside of a Databricks environment without significant refactoring of these integrations. To view the logic, please read the `.py`/`.ipynb` files directly.

#### Notes on statistical methodology

-  **Normalisation:** When grouping by demographic, the average spend and average purchases are normalised to **monthly rates** (`SpendPerMonth`, `PurchasesPerMonth`) to account for varying customer tenure. This prevents bias toward longstanding customers

-  **Aggregation:** Similarly, When grouping by demographic, average ratios (eg channel mix ratios) report the **average of per-customer ratios** (eg, `AVG(PercentDealPurchases)`) rather than the aggregate ratio of sums (`SUM(Deals) / SUM(Total)`). **Aggregate of sums** weights heavily toward high-volume/longstanding customers, masking the behaviour of the "typical" customer in a segment. **Average of ratios** answers the question *"What is the average deal-seeking behaviour of a customer in this segment?"* 
  
**Caveat:** This approach can be misleading if segment sizes vary wildly (eg, one customer in a segment). In this dataset, all segments have meaningful volume, making the "average of ratios" the more actionable metric for marketing strategy.

- **RFM values**: *Recency* is between 1-99 days and is scored between 1-3 inclusive. *Monetary* is the total of the last two years of spend and is scored between 1-5 inclusive.  *Frequency* is the number of purchases in the last two years and is scored between 1-5 inclusive. **Please see the [Marketing project](https://github.com/datagranate/data-portfolio/blob/main/marketing/README.md#rfm-methodology-and-limitations) for a additional details and discussion.** 

In [0]:
# set variables for catalog, schema, volume names 
catalog_name = "analytics"
schema_name = "marketing"
volume_name = "data_sources"

Create SQL query functions grouping by four demographic categories, `Education`, `Marital_Status`, `Age_category`, `Income_category`

In [0]:
tbl = f"{catalog_name}.{schema_name}.customer_behaviour_summary"

ALLOWED_DEMO = ["Education", "Marital_Status", "Age_category", "Income_category"]

def create_param_functions(catalog_name, schema_name):
    # demographic profile 
    spark.sql(f"""
        CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.demographic_profile(demo_field STRING COMMENT 'Customer demographic field')
        RETURNS TABLE (
            group_value STRING COMMENT 'demographic group value',
            avg_spend_per_month DOUBLE,
            avg_purchases_per_month DOUBLE,
            avg_recency_days DOUBLE COMMENT 'Mean days since last purchase; higher = less recently active'
        )
        COMMENT 'Returns average monthly spend, avg count of purchases and average recency grouped by a customer demographic. The customer demographic to group by MUST be copied verbatim (case-sensitive) from exactly one of: "Education", "Marital_Status", "Age_category", "Income_category". Never abbreviate or paraphrase, eg use "Income_category" not "income", "Age_category" not "age".'
        LANGUAGE SQL
        RETURN (
            SELECT
                CASE demo_field
                    WHEN 'Education'       THEN CAST(Education AS STRING)
                    WHEN 'Marital_Status'  THEN CAST(Marital_Status AS STRING)
                    WHEN 'Age_category'    THEN CAST(Age_category AS STRING)
                    WHEN 'Income_category' THEN CAST(Income_category AS STRING)
                END AS group_value,
                ROUND(AVG(SpendPerMonth), 2),
                ROUND(AVG(PurchasesPerMonth), 2),
                ROUND(CAST(AVG(Recency) AS DOUBLE), 2)
            FROM {tbl}
            WHERE demo_field IN ('Education','Marital_Status','Age_category','Income_category')
            GROUP BY 1
            ORDER BY group_value
        );
    """)

    # channel mix by demographic
    spark.sql(f"""
        CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.channel_mix_by_demographic(demo_field STRING COMMENT 'Customer demographic field')
        RETURNS TABLE (
            group_value STRING COMMENT 'demographic group value',
            avg_web_ratio DOUBLE,
            avg_store_ratio DOUBLE,
            avg_catalogue_ratio DOUBLE
        )
        COMMENT 'Returns average web / store / catalogue purchase ratios grouped by a customer demographic. Ratios are proportions between 0 and 1. The customer demographic to group by MUST be copied verbatim (case-sensitive) from exactly one of: "Education", "Marital_Status", "Age_category", "Income_category". Never abbreviate or paraphrase, eg use "Income_category" not "income", "Age_category" not "age".'
        LANGUAGE SQL
        RETURN (
            SELECT
                CASE demo_field
                    WHEN 'Education'       THEN CAST(Education AS STRING)
                    WHEN 'Marital_Status'  THEN CAST(Marital_Status AS STRING)
                    WHEN 'Age_category'    THEN CAST(Age_category AS STRING)
                    WHEN 'Income_category' THEN CAST(Income_category AS STRING)
                END AS group_value,
                ROUND(AVG(NumWebPurchases / NULLIF(NumPurchases, 0)), 4),
                ROUND(AVG(NumStorePurchases / NULLIF(NumPurchases, 0)), 4),
                ROUND(AVG(NumCatalogPurchases / NULLIF(NumPurchases, 0)), 4)
            FROM {tbl}
            WHERE demo_field IN ('Education','Marital_Status','Age_category','Income_category')
            GROUP BY 1
            ORDER BY group_value
        );
    """)

    # deal affinity by demographic
    spark.sql(f"""
        CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.deal_affinity_by_demographic(demo_field STRING COMMENT 'Customer demographic field')
        RETURNS TABLE (
            group_value STRING COMMENT 'demographic group value',
            avg_deal_ratio DOUBLE,
            avg_deal_count DOUBLE
        )
        COMMENT 'Returns average deal ratio and average number of purchases using deals, grouped by a customer demographic. The deal ratio is a proportion between 0 and 1. The customer demographic to group by MUST be copied verbatim (case-sensitive) from exactly one of: "Education", "Marital_Status", "Age_category", "Income_category". Never abbreviate or paraphrase, eg use "Income_category" not "income", "Age_category" not "age".'
        LANGUAGE SQL
        RETURN (
            SELECT
                CASE demo_field
                    WHEN 'Education'       THEN CAST(Education AS STRING)
                    WHEN 'Marital_Status'  THEN CAST(Marital_Status AS STRING)
                    WHEN 'Age_category'    THEN CAST(Age_category AS STRING)
                    WHEN 'Income_category' THEN CAST(Income_category AS STRING)
                END AS group_value,
                ROUND(AVG(PercentDealPurchases), 4),
                ROUND(CAST(AVG(NumDealsPurchases) AS DOUBLE), 4)
            FROM {tbl}
            WHERE demo_field IN ('Education','Marital_Status','Age_category','Income_category')
            GROUP BY 1
            ORDER BY group_value
        );
    """)

create_param_functions(catalog_name, schema_name)


Create SQL functions using RFM categories and scores.  Segments were defined in the Marketing project notebook [marketing_rfm.ipynb](https://github.com/datagranate/data-portfolio/blob/main/marketing/notebooks/marketing_rfm.ipynb). 

In [0]:
# RFM segment queries (segments already defined in the main Marketing project)

VALID_SEGMENTS = ["Champions", "High-potential growth", "Loyal", "Recents",
                  "Critical retention", "Low-priority churn", "High value at risk", "Standard base"]
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.rfm_segment_summary()
    RETURNS TABLE (
        Segment STRING COMMENT 'RFM segment name',
        customer_count INT,
        avg_recency_days DOUBLE,
        avg_frequency DOUBLE,
        avg_monetary DOUBLE,
        avg_r_score DOUBLE,
        avg_f_score DOUBLE,
        avg_m_score DOUBLE
    )
    LANGUAGE SQL
    COMMENT 'Returns a table of RFM segment names and summary statistics for each segment.'
    RETURN (
        SELECT
            Segment,
            CAST(COUNT(*) AS INT) AS customer_count,
            ROUND(CAST(AVG(Recency) AS DOUBLE), 2) AS avg_recency_days,
            ROUND(CAST(AVG(Frequency) AS DOUBLE), 2) AS avg_frequency,
            ROUND(CAST(AVG(Monetary) AS DOUBLE), 2) AS avg_monetary,
            ROUND(CAST(AVG(R) AS DOUBLE), 2) AS avg_r_score,
            ROUND(CAST(AVG(F) AS DOUBLE), 2) AS avg_f_score,
            ROUND(CAST(AVG(M) AS DOUBLE), 2) AS avg_m_score
        FROM {tbl}
        GROUP BY Segment
    );
""")


# customer-level lookup by segment
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.rfm_customers_in_segment(segment_name STRING COMMENT 'Customer RFM segment')
    RETURNS TABLE (
        customer_id INT,
        Segment STRING,
        R INT,
        F INT,
        M INT,
        RFM_score STRING,
        Recency DOUBLE,
        Frequency DOUBLE,
        Monetary DOUBLE,
        Age_category STRING,
        Income_category STRING,
        Marital_Status STRING,
        Education STRING
    )
    LANGUAGE SQL
    COMMENT 'Returns individual customers belonging to one named RFM segment, with their R/F/M values and scores and demographics. The segment_name MUST be copied verbatim from exactly one of: "Champions", "High-potential growth", "Loyal", "Recents", "Critical retention", "Low-priority churn", "High value at risk", "Standard base"'
    RETURN (
        SELECT
            ID AS customer_id,
            Segment,
            CAST(R AS INT) AS R,
            CAST(F AS INT) AS F,
            CAST(M AS INT) AS M,
            CAST(RFM_score AS STRING) AS RFM_score,
            CAST(Recency AS DOUBLE) AS Recency,
            CAST(Frequency AS DOUBLE) AS Frequency,
            CAST(Monetary AS DOUBLE) AS Monetary,
            CAST(Age_category AS STRING) AS Age_category,
            CAST(Income_category AS STRING) AS Income_category,
            CAST(Marital_Status AS STRING) AS Marital_Status,
            CAST(Education AS STRING) AS Education
        FROM {tbl}
        WHERE segment_name IN {tuple(VALID_SEGMENTS)}
            AND Segment = segment_name
        DISTRIBUTE BY (1)
    );
""")

# score-band lookup: returns all customers whose individual R, F or M score matches a value
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.rfm_customers_by_score_band(
        score_component STRING COMMENT 'The RFM component to filter by, ie "R", "F" or "M',
        score_value INT COMMENT 'The RFM component score to filter by'
    )
    RETURNS TABLE (
        customer_id STRING,
        Segment STRING,
        R INT,
        F INT,
        M INT,
        RFM_score STRING,
        Recency DOUBLE,
        Frequency DOUBLE,
        Monetary DOUBLE
    )
    LANGUAGE SQL
    COMMENT 'Returns individual customers whose R, F or M score matches a value. The score_component MUST be copied verbatim from exactly one of: "R", "F", "M". The score_value for "R" must be between 1 and 3 inclusive. The score_value for "F" and "M" must be between 1 and 5 inclusive.'
    RETURN (
        SELECT
            ID AS customer_id,
            Segment,
            CAST(R AS INT) AS R,
            CAST(F AS INT) AS F,
            CAST(M AS INT) AS M,
            CAST(RFM_score AS STRING) AS RFM_score,
            CAST(Recency AS DOUBLE) AS Recency,
            CAST(Frequency AS DOUBLE) AS Frequency,
            CAST(Monetary AS DOUBLE) AS Monetary
        FROM {tbl}
        WHERE score_component IN ('R', 'F', 'M')
          AND (
              (score_component = 'R' AND score_value BETWEEN 1 AND 3 AND R = score_value)
           OR (score_component = 'F' AND score_value BETWEEN 1 AND 5 AND F = score_value)
           OR (score_component = 'M' AND score_value BETWEEN 1 AND 5 AND M = score_value)
           )
        DISTRIBUTE BY (1)
    );
""")

spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.channel_mix_by_rfm_segment()
    RETURNS TABLE (
        Segment STRING,
        avg_web_ratio DOUBLE,
        avg_store_ratio DOUBLE,
        avg_catalogue_ratio DOUBLE
    )
    COMMENT 'Returns the average channel mix for each RFM segment. The segment_name MUST be copied verbatim from exactly one of: "Champions", "High-potential growth", "Loyal", "Recents", "Critical retention", "Low-priority churn", "High value at risk", "Standard base"'
    LANGUAGE SQL
    RETURN (
        SELECT
            Segment,
            ROUND(AVG(NumWebPurchases / NULLIF(NumPurchases, 0)), 4),
            ROUND(AVG(NumStorePurchases / NULLIF(NumPurchases, 0)), 4),
            ROUND(AVG(NumCatalogPurchases / NULLIF(NumPurchases, 0)), 4)
        FROM {tbl}
        GROUP BY Segment
    );
""")


DataFrame[]

Test queries

In [0]:
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.demographic_profile('Age_category');").display() 

group_value,avg_spend_per_month,avg_purchases_per_month,avg_recency_days
18-24,53.56,1.0,45.25
25-34,42.35,0.94,48.54
35-44,40.31,1.01,48.48
45-54,47.95,1.11,50.38
55-64,50.99,1.18,50.94
65-74,82.06,1.5,46.48


In [0]:
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.channel_mix_by_demographic('Income_category');").display() 

group_value,avg_web_ratio,avg_store_ratio,avg_catalogue_ratio
High,0.2779,0.4306,0.2915
Low,0.3423,0.5907,0.0671
Medium,0.3514,0.5029,0.1456


In [0]:
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.deal_affinity_by_demographic('Marital_Status');").display() 

group_value,avg_deal_ratio,avg_deal_count
Divorced,0.2448,2.4251
Married,0.243,2.3907
Single,0.2394,2.1267
Together,0.243,2.3105
Widow,0.2036,2.2319


In [0]:
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.rfm_segment_summary();").display() 

Segment,customer_count,avg_recency_days,avg_frequency,avg_monetary,avg_r_score,avg_f_score,avg_m_score
High value at risk,263,50.54,19.94,1177.8,2.0,4.3,4.51
Standard base,402,49.2,7.8,155.27,2.0,2.0,1.99
Champions,224,16.33,20.81,1167.47,3.0,4.47,4.52
Recents,274,15.99,5.2,59.11,3.0,1.42,1.51
Low-priority churn,390,82.74,7.67,148.33,1.0,1.97,2.0
Critical retention,274,82.86,19.41,1176.17,1.0,4.23,4.48
Loyal,106,16.35,12.33,320.14,3.0,3.0,3.0
High-potential growth,75,14.65,15.28,914.4,3.0,3.48,3.83


In [0]:
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.rfm_customers_in_segment('Champions');").display() 

customer_id,Segment,R,F,M,RFM_score,Recency,Frequency,Monetary,Age_category,Income_category,Marital_Status,Education
4141,Champions,3,4,4,344,26.0,20.0,734.0,45-54,High,Together,Graduation
7446,Champions,3,4,4,344,16.0,20.0,702.0,45-54,Medium,Together,Master
2114,Champions,3,5,5,355,23.0,25.0,1270.0,65-74,High,Single,PhD
8755,Champions,3,4,5,345,4.0,17.0,1174.0,65-74,High,Married,Master
8601,Champions,3,4,5,345,3.0,19.0,1293.0,25-34,High,Married,Graduation
4114,Champions,3,5,5,355,2.0,28.0,1661.0,45-54,High,Married,Master
10383,Champions,3,4,4,344,30.0,20.0,1053.0,45-54,High,Divorced,Graduation
6853,Champions,3,4,5,345,12.0,20.0,1358.0,25-34,High,Single,Master
3076,Champions,3,4,4,344,18.0,18.0,606.0,35-44,Medium,Together,Graduation
1012,Champions,3,4,4,344,26.0,20.0,715.0,55-64,Medium,Single,Graduation


In [0]:
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.rfm_customers_by_score_band('R', 1);").display() 

customer_id,Segment,R,F,M,RFM_score,Recency,Frequency,Monetary
5324,Low-priority churn,1,3,3,133,94.0,14.0,407.0
5899,Low-priority churn,1,1,1,111,68.0,1.0,36.0
2125,Critical retention,1,3,4,134,82.0,15.0,1072.0
6565,Critical retention,1,5,5,155,91.0,24.0,1606.0
2278,Low-priority churn,1,2,2,122,86.0,6.0,94.0
1409,Low-priority churn,1,3,3,133,69.0,13.0,345.0
7892,Low-priority churn,1,2,2,122,89.0,6.0,62.0
1966,Critical retention,1,5,5,155,96.0,25.0,1672.0
10755,Low-priority churn,1,2,2,122,76.0,6.0,45.0
8595,Low-priority churn,1,1,2,112,99.0,5.0,63.0


In [0]:
spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.channel_mix_by_rfm_segment();").display() 

Segment,avg_web_ratio,avg_store_ratio,avg_catalogue_ratio
High value at risk,0.2948,0.4375,0.2678
Standard base,0.3526,0.5568,0.0905
Champions,0.2978,0.4357,0.2666
Recents,0.3303,0.6011,0.0686
Low-priority churn,0.3541,0.5477,0.0982
Critical retention,0.2926,0.4351,0.2723
Loyal,0.401,0.4606,0.1384
High-potential growth,0.3415,0.4325,0.226
